# 05. 승차 변화율 지도

**원본**: `Analisis3/002_morning_change_grid_map_250m.ipynb` (증감량 지도)

증감량 지도는 원래 승차가 많은 셀만 진하게 보인다. 그래서 **변화율(%)**로 다시 그렸다.
- 2024년 승차 5명 미만 셀은 변화율이 크게 튀므로 회색으로 표시한다.
- 노란 점 = GTX-A역. 레이어 버튼으로 오전/종일을 전환할 수 있다.
- 결과 파일: `output/map_change_rate.html` (브라우저로 열기)

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats

data = load_model_data()  # data/processed/09_회귀분석_격자통합데이터.csv

In [2]:
import folium
from branca.colormap import LinearColormap

grid = stable_grid_polygons().merge(
    data[["셀 ID", "board_2024", "board_2025", "board_2024_all", "board_2025_all", "walk_min", "nearest_station"]], on="셀 ID")
grid["오전 변화율(%)"] = np.where(grid.board_2024 >= 5, (grid.board_2025 / grid.board_2024.where(grid.board_2024 > 0) - 1) * 100, np.nan)
grid["종일 변화율(%)"] = np.where(grid.board_2024_all >= 5, (grid.board_2025_all / grid.board_2024_all.where(grid.board_2024_all > 0) - 1) * 100, np.nan)
grid = grid.to_crs("EPSG:4326")

colormap = LinearColormap(["#2166AC", "#F7F7F7", "#B2182B"], vmin=-100, vmax=100)
colormap.caption = "승차인원 변화율(%) 2024→2025 (파랑=감소, 회색=2024년 5명 미만)"
fmap = folium.Map(location=[37.70, 126.78], zoom_start=11, tiles="CartoDB positron", control_scale=True)
for column, show in [("오전 변화율(%)", True), ("종일 변화율(%)", False)]:
    folium.GeoJson(
        grid.round(1).to_json(), name=column, show=show,
        style_function=lambda f, c=column: {
            "fillColor": "#BBBBBB" if f["properties"][c] is None else colormap(max(-100, min(100, f["properties"][c]))),
            "color": "#333333", "weight": 0.6, "fillOpacity": 0.8},
        tooltip=folium.GeoJsonTooltip(
            fields=["셀 ID", "nearest_station", "walk_min", "board_2024", "board_2025", "오전 변화율(%)", "board_2024_all", "board_2025_all", "종일 변화율(%)"],
            aliases=["셀 ID", "최근접 GTX역", "보행시간(분)", "2024 오전", "2025 오전", "오전 변화율(%)", "2024 종일", "2025 종일", "종일 변화율(%)"]),
    ).add_to(fmap)
for name, (lat, lon) in GTX_STATIONS.items():
    folium.CircleMarker([lat, lon], radius=7, color="black", fill=True, fill_color="#FFD700", fill_opacity=1, tooltip=f"GTX-A {name}역").add_to(fmap)
colormap.add_to(fmap)
folium.LayerControl(collapsed=False).add_to(fmap)
fmap.save(MAP_PATH)
print(f"저장: {MAP_PATH.name}")
fmap

저장: map_change_rate.html
